# 04 Compare all decoders
## Цель ноутбука
Главный демонстрационный ноутбук: сравнение Viterbi, BCJR, Neural Viterbi и Neural BCJR.

## Что мы получим
Таблицу, графики и автоматические выводы по качеству/скорости.


In [ ]:
# [1/8] Импорты и поиск корня проекта
from pathlib import Path
import sys
import yaml
import pandas as pd
from IPython.display import display, Image

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    while cur != cur.parent:
        if (cur / 'pyproject.toml').exists():
            return cur
        cur = cur.parent
    raise RuntimeError('Repo root not found')

ROOT = find_repo_root(Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print('[1/8] Импорты загружены')
print('ROOT =', ROOT)


# Настройка параметров эксперимента


In [ ]:
# [2/8] Параметры эксперимента (редактируются в этой ячейке)
PARAMS = {
    'run_name': 'notebook_compare_all',
    'K': 64,
    'num_blocks': 20,
    'snr_db_list': [0, 1, 2, 3, 4],
    'seed': 123,
    'decoders': ['viterbi', 'bcjr', 'neural_viterbi', 'neural_bcjr'],
    'epochs': 3,
    'learning_rate': 1e-3,
    'hidden_dim': 16,
    'reuse_saved_signals': False,
    'training_enabled': True,
}
print('[2/8] Параметры заданы')
display(pd.Series(PARAMS))


In [ ]:
# [3/8] Загружаем YAML, меняем параметры и сохраняем notebook-конфиг
from comm_ai.utils.io import load_yaml

cfg = load_yaml(ROOT / 'src/comm_ai/config/experiments/awgn_small.yaml')
cfg['experiment']['run_name'] = PARAMS['run_name']
cfg['experiment']['K'] = PARAMS['K']
cfg['experiment']['num_blocks'] = PARAMS['num_blocks']
cfg['experiment']['snr_db_list'] = PARAMS['snr_db_list']
cfg['experiment']['seed'] = PARAMS['seed']
cfg['experiment']['decoders'] = PARAMS['decoders']
cfg['experiment']['reuse_saved_signals'] = PARAMS['reuse_saved_signals']

cfg['training']['enabled'] = PARAMS['training_enabled']
cfg['training']['epochs'] = PARAMS['epochs']
cfg['training']['learning_rate'] = PARAMS['learning_rate']
cfg['training']['hidden_dim'] = PARAMS['hidden_dim']

nb_cfg_path = ROOT / 'outputs/runs' / f"{PARAMS['run_name']}_notebook_config.yaml"
nb_cfg_path.parent.mkdir(parents=True, exist_ok=True)
nb_cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('[3/8] Конфиг сохранён в', nb_cfg_path)


### Пояснение
Для более гладких BER/FER-кривых используйте больше точек SNR.


In [ ]:
# [4/8] Пример сигнального pipeline: u -> c -> x -> noise -> y -> llr
from comm_ai.datasets.signals_dataset import SignalsDataset

ds_preview = SignalsDataset.generate(cfg)
print('[4/8] Preview dataset сгенерирован')
print('u[:16]     =', ds_preview.u[0][:16])
print('c[:16]     =', ds_preview.c[0][:16])
print('x[:16]     =', ds_preview.x[0][:16])
print('noise[:16] =', ds_preview.noise[0][:16])
print('y[:16]     =', ds_preview.y[0][:16])
print('llr[:16]   =', ds_preview.llr[0][:16])
print('Комментарий: чем меньше SNR, тем сильнее шум и сложнее декодирование.')


In [ ]:
# [5/8] Запуск эксперимента
from comm_ai.experiments.run_experiment import run
out_dir = run(str(nb_cfg_path))
print('[5/8] Эксперимент завершён, артефакты в', out_dir)


In [ ]:
# [6/8] Таблица результатов и авто-анализ
from comm_ai.utils.reporting import analyze_results

results = pd.read_csv(out_dir / 'results.csv')
display(results)
a = analyze_results(results)
print('[6/8] Лучший BER:', a['best_ber'])
print('[6/8] Лучший FER:', a['best_fer'])
print('[6/8] Самый быстрый:', a['fastest'])
print('[6/8] Интерпретация:', a['tradeoff'])


In [ ]:
# [7/8] Графики с пояснением
print('График BER')
print('X: SNR [dB] - отношение мощности сигнала к мощности шума')
print('Y: BER - доля ошибочно восстановленных битов')
display(Image(filename=str(out_dir / 'ber_plot.png')))

print('График FER')
print('X: SNR [dB] - отношение мощности сигнала к мощности шума')
print('Y: FER - доля блоков с хотя бы одной ошибкой')
display(Image(filename=str(out_dir / 'fer_plot.png')))

print('График времени декодирования')
print('X: SNR [dB] - отношение мощности сигнала к мощности шума')
print('Y: Decode time [s] - среднее время декодирования в секундах')
display(Image(filename=str(out_dir / 'timing_plot.png')))

if 'results' in globals():
    grouped = results.groupby('decoder', as_index=False).agg(ber=('ber','mean'), fer=('fer','mean'), t=('decode_time_s','mean'))
    print('Краткий вывод по графикам:')
    print('- лучший BER:', grouped.loc[grouped['ber'].idxmin(), 'decoder'])
    print('- лучший FER:', grouped.loc[grouped['fer'].idxmin(), 'decoder'])
    print('- самый быстрый:', grouped.loc[grouped['t'].idxmin(), 'decoder'])


## Итоговый вывод
Используйте этот ноутбук как основной для демонстрации в дипломе.


## Итог исследования
Ниже формируется итог по качеству и скорости на основе `results.csv`.


In [ ]:
from comm_ai.utils.reporting import analyze_results
final_analysis = analyze_results(results)
print('Лучший алгоритм по BER:', final_analysis['best_ber'])
print('Лучший алгоритм по FER:', final_analysis['best_fer'])
print('Самый быстрый алгоритм:', final_analysis['fastest'])
print('Компромисс качество/время:', final_analysis['tradeoff'])
print('Краткая интерпретация: выбор декодера зависит от приоритета между качеством восстановления и временем декодирования.')
